# Elastic Net Regresyonu

## Temel Kavramlar

Elastic Net, Ridge (L2) ve LASSO (L1) regresyonlarının güçlü yönlerini birleştiren bir düzenlileştirme (regularization) yöntemidir. Standart çoklu doğrusal regresyonda model, tahmin edilen ve gerçek değerler arasındaki hata kareleri toplamını (SSE) minimize etmeye çalışır. Ancak, model çok karmaşık olduğunda veya çok sayıda yüksek korelasyonlu özellik (multicollinearity) bulunduğunda aşırı öğrenme (overfitting) meydana gelebilir.

Elastic Net, amaç fonksiyonuna hem L1 hem de L2 cezalarını ekleyerek bu sorunu çözer. Bu sayede modelin karmaşıklığı kontrol altına alınır.

---

## Amaç Fonksiyonu

Elastic Net'in minimize etmeye çalıştığı amaç fonksiyonu şu şekildedir:

$$\min_{\beta} \left\{ \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \lambda_1 \sum_{j=1}^{p} |\beta_j| + \lambda_2 \sum_{j=1}^{p} \beta_j^2 \right\}$$

Sembollerin anlamı:
- $\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$ → Hata Kareleri Toplamı (Veriye uyum terimi)
- $\lambda_1 \sum_{j=1}^{p} |\beta_j|$ → L1 Ceza Terimi (Katsayıları tam sıfıra çeker, seyreklik sağlar)
- $\lambda_2 \sum_{j=1}^{p} \beta_j^2$ → L2 Ceza Terimi (Katsayıların aşırı büyümesini engeller, korelasyonlu özelliklerde kararlılık sağlar)

Matris ve norm notasyonu kullanılarak daha sade bir biçimde yazılabilir:

$$\min_{\beta} \left\{ \|y - X\beta\|_2^2 + \lambda_1 \|\beta\|_1 + \lambda_2 \|\beta\|_2^2 \right\}$$

Burada:
- $\|\beta\|_1$, L1 normudur (Katsayıların mutlak değerlerinin toplamı).
- $\|\beta\|_2^2$, karesi alınmış L2 normudur (Katsayıların karelerinin toplamı).

---

## Neden İkisi Birlikte Kullanılır?

- **L1 Cezası (LASSO):** Modelde özellik seçimi yapar. Bazı katsayıları tam olarak sıfıra indirger. Ancak yüksek korelasyonlu özellik grupları arasından rastgele yalnızca birini seçme eğilimindedir.
- **L2 Cezası (Ridge):** Korelasyonlu özelliklere karşı daha kararlıdır (gruplama etkisi). Katsayıları küçültür ancak hiçbir zaman tam sıfır yapmaz.
- **Elastic Net:** L1 ve L2'nin birleşimi, hem seyreklik (özellik seçimi) sağlar hem de korelasyonlu özellikleri birlikte modelde tutarak istikrar kazandırır.

#

## Geometrik Yorum

Kısıt (constraint) bölgeleri geometrik olarak incelendiğinde yöntemler arasındaki fark netleşir.

- **LASSO (L1 - Elmas Şekli):** Kısıt bölgesinin köşeleri eksenler üzerindedir. Hata yüzeyi bu köşelere çarptığında katsayılar tam sıfır olur.
- **Ridge (L2 - Daire Şekli):** Köşesi yoktur, pürüzsüzdür. Katsayılar sıfıra yaklaşır ancak tam sıfır olmaz.
- **Elastic Net (Yuvarlatılmış Elmas):** L1 ve L2 kısıtlarının birleşimi, köşeleri olan ancak kenarları dışa doğru kavisli bir şekil oluşturur. Bu yapı, hem eksenlerde (sıfır katsayı) çözüm bulunmasına hem de pürüzsüz kenarlar sayesinde korelasyonlu özelliklerin bir arada tutulmasına olanak tanır.

#

## Adım Adım Matris Hesaplaması (Elle Çözüm Örneği)

Elastic Net algoritmasının çalışma mantığının anlaşılması için 4 gözlem ($n=4$) ve 3 özellikten ($p=3$) oluşan örnek bir veri seti üzerinden matris işlemleri incelenmelidir.

Özellik 1 ve Özellik 3 yüksek düzeyde korelasyonludur (Özellik 3 $\approx$ Özellik 1 + 0.1). L1 parametresi $\lambda_1 = 0.1$ ve L2 parametresi $\lambda_2 = 0.05$ olarak belirlenmiştir.

$$X = \begin{bmatrix} 1 & 2 & 1.1 \\ 2 & 3 & 2.1 \\ 3 & 1 & 3.1 \\ 4 & 2 & 4.1 \end{bmatrix}, \quad y = \begin{bmatrix} 5 \\ 8 \\ 6 \\ 10 \end{bmatrix}$$

### Adım 1: Özelliklerin Ölçeklendirilmesi (Standardization)
Düzenlileştirme yöntemleri özelliklerin ölçeklerine karşı hassastır. Büyük değerli özelliklerin daha fazla cezalandırılmasını önlemek için tüm özellikler standartlaştırılmalıdır (Ortalama = 0, Varyans = 1).

Standartlaştırma formülü:
$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$

Özellik 1 için ($x_1 = [1, 2, 3, 4]$):
- Ortalama: $\bar{x}_1 = 2.5$
- Standart Sapma: $s_1 \approx 1.291$

Tüm özellikler standartlaştırıldığında elde edilen $X_{std}$ matrisi:
$$X_{std} \approx \begin{bmatrix} -1.34 & 0.00 & -1.34 \\ -0.45 & 1.41 & -0.45 \\ 0.45 & -1.41 & 0.45 \\ 1.34 & 0.00 & 1.34 \end{bmatrix}$$

### Adım 2: Kovaryans Matrisinin Hesaplanması ($X_{std}^T X_{std}$)
Standartlaştırılmış matrisin transpozu ile çarpımı, özellikler arasındaki ilişkileri (korelasyonu) ortaya çıkarır.

$$X_{std}^T X_{std} = \begin{bmatrix} 4.0 & -1.26 & 4.0 \\ -1.26 & 4.0 & -1.26 \\ 4.0 & -1.26 & 4.0 \end{bmatrix}$$

**Matrisin Yorumu:**
- $(1,1)$ ve $(3,3)$ köşegen elemanları $4.0$'dır.
- $(1,3)$ ve $(3,1)$ dış elemanları da $4.0$'dır. Bu durum, Özellik 1 ile Özellik 3'ün birbiriyle kusursuz korelasyona sahip olduğunu gösterir.

### Adım 3: L2 Cezasının Eklenmesi ($X_{std}^T X_{std} + \lambda_2 I$)
Ridge (L2) etkisini sağlamak için köşegen elemanlara $\lambda_2$ değeri (burada $0.05$) eklenir. Bu işlem, matrisin tersinin alınabilirliğini garanti altına alır ve katsayı varyansını düşürür.

$$X_{std}^T X_{std} + \lambda_2 I = \begin{bmatrix} 4.05 & -1.26 & 4.0 \\ -1.26 & 4.05 & -1.26 \\ 4.0 & -1.26 & 4.05 \end{bmatrix}$$

### Adım 4: $X^T y$ Matrisinin Hesaplanması
Özelliklerin hedef değişken ile olan ilişkisi hesaplanır.

$$X_{std}^T y = \begin{bmatrix} -1.34 & -0.45 & 0.45 & 1.34 \\ 0.00 & 1.41 & -1.41 & 0.00 \\ -1.34 & -0.45 & 0.45 & 1.34 \end{bmatrix} \begin{bmatrix} 5 \\ 8 \\ 6 \\ 10 \end{bmatrix} = \begin{bmatrix} 5.8 \\ 2.82 \\ 5.8 \end{bmatrix}$$
Burada 1. ve 3. elemanların ($5.8$) aynı olması, yine yüksek korelasyonun sonucudur.

### Adım 5: Optimizasyon ve Çözüm
Ridge regresyonunun aksine, Elastic Net'in mutlak değer içeren L1 cezası nedeniyle kapalı form (closed-form) analitik bir çözümü yoktur. Denklem türevlenemez noktalar barındırır.

Bu nedenle çözüm, **Coordinate Descent (Koordinat İnişi)** algoritması ile iteratif olarak bulunur. Algoritma her adımda L1 cezası için **Soft-Thresholding** uygulayarak katsayıları günceller. Optimizasyon sonucunda korelasyonlu Özellik 1 ve Özellik 3 modelde birbirine yakın değerlerle tutulurken, L1 etkisinden dolayı ağırlığı düşük olan özellikler sıfırlanabilir.

#

## Scikit-learn ile Elastic Net Uygulaması

Scikit-learn kütüphanesindeki formülasyon şu parametreleri kullanır:
- `alpha`: Toplam düzenlileştirme gücü (matematiksel notasyondaki $\lambda$)
- `l1_ratio`: L1 ve L2 arasındaki denge (matematiksel notasyondaki $\alpha$ veya $\rho$). $1$ tam LASSO, $0$ tam Ridge anlamına gelir.

#

In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Tekrarlanabilirlik için seed ayarı
np.random.seed(42)

# Sentetik veri seti üretimi
X, y = make_regression(n_samples=1000, n_features=20, noise=50.0, random_state=42)

# Elastic Net'in "gruplama" etkisini gözlemlemek için yapay korelasyonlu özellikler oluşturulması
# Özellik 0 ve 1 yüksek korelasyonlu
X[:, 1] = X[:, 0] + 0.3 * np.random.randn(1000)
# Özellik 2 ve 3 yüksek korelasyonlu
X[:, 3] = 0.7 * X[:, 2] + 0.3 * np.random.randn(1000)

# Verinin eğitim ve test kümelerine bölünmesi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ElasticNetCV: Cross-validation ile en iyi alpha ve l1_ratio değerlerini otomatik bulur.
elastic_net = ElasticNetCV(
    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], # Denge parametresi
    alphas=np.logspace(-3, 1, 20),      # Toplam ceza katsayısı gücü aralığı
    cv=5,                               # 5 katlı çapraz doğrulama
    max_iter=2000,                      # İterasyon limiti
    random_state=42,
    n_jobs=-1                           # Tüm işlemci çekirdeklerinin kullanılması
)

# Pipeline: Veri sızıntısını (data leakage) önlemek için önce ölçeklendirme, sonra modelleme
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic_net", elastic_net)
])

# Modelin eğitilmesi
pipeline.fit(X_train, y_train)

# Tahmin ve Metrikler
y_pred = pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

best_model = pipeline.named_steps["elastic_net"]

print("--- Performans Metrikleri ---")
print(f"MSE: {mse:.4f}")
print(f"R² Skoru: {r2:.4f}\n")

print("--- Optimal Hiperparametreler ---")
print(f"En iyi l1_ratio: {best_model.l1_ratio_:.3f}")
print(f"En iyi alpha: {best_model.alpha_:.4f}\n")

print("--- Özellik Seçimi Sonuçları ---")
non_zero_coefs = 0
for i, coef in enumerate(best_model.coef_):
    if abs(coef) > 1e-6:
        non_zero_coefs += 1
        print(f"Özellik {i:2d}: {coef:8.4f}")

print(f"\nSeçilen özellik sayısı: {non_zero_coefs} / {X.shape[1]}")

--- Performans Metrikleri ---
MSE: 12234.5750
R² Skoru: 0.7076

--- Optimal Hiperparametreler ---
En iyi l1_ratio: 0.900
En iyi alpha: 0.1274

--- Özellik Seçimi Sonuçları ---
Özellik  0:  81.9982
Özellik  1:   4.0983
Özellik  2:   6.4044
Özellik  4:  83.1459
Özellik  5:  -0.5909
Özellik  6:  69.7527
Özellik  7:   7.5425
Özellik  8:  -1.2363
Özellik  9:   3.9866
Özellik 10:  19.8315
Özellik 11:  44.8573
Özellik 12:   3.7041
Özellik 13:   2.3340
Özellik 14:  -3.9652
Özellik 15:  24.8593
Özellik 16:  -0.3934
Özellik 17:  83.3584
Özellik 18:   2.5290
Özellik 19:   1.7641

Seçilen özellik sayısı: 19 / 20


## Avantajlar, Dezavantajlar ve Karşılaştırma

| Özellik | Açıklama |
| :--- | :--- |
| **Avantaj** | Hem özellik seçimi (L1) yapar hem de korelasyonlu değişkenlerde kararlıdır (L2). |
| **Avantaj** | Özellik sayısı ($p$), gözlem sayısından ($n$) büyük olduğunda ($p > n$) LASSO'dan daha iyi performans gösterir. |
| **Dezavantaj** | İki farklı hiperparametrenin (`alpha` ve `l1_ratio`) optimize edilmesi gerektiği için hesaplama maliyeti yüksektir. |
| **Dezavantaj** | Saf LASSO'ya kıyasla daha az seyreklik (daha az katsayıyı sıfırlama) gösterebilir, bu da çok yüksek boyutlarda yorumlanabilirliği zorlaştırabilir. |

### Yöntemlerin Karşılaştırması

| Özellik | Ridge (L2) | LASSO (L1) | Elastic Net (L1 + L2) |
| :--- | :--- | :--- | :--- |
| **Katsayı Sıfırlama** | Yok (Sadece küçültür) | Var | Var |
| **Korelasyonlu Veri Yaklaşımı** | Özellikleri bir arada tutar | Birini seçer, diğerlerini eler | Bir arada tutar |
| **Kapalı Form Çözüm** | Var (Matris tersi) | Yok (İteratif) | Yok (İteratif) |
| **Kullanım Senaryosu** | Tüm özellikler önemliyse | Çoğu özellik gereksizse | Hem gereksiz özellik var hem de yüksek korelasyon varsa |